# Adaptation Potential Metrics

This notebook creates the Adaptation Potential section CSV for the national tool. It summarizes existing FLOPROS flood-protection standards and nature-based solution opportunity areas, implementation costs, carbon benefits, and biodiversity benefits.

## 0. Setup

Country settings, administrative level, source paths, nominal NbS cell area, and the coastal assignment limit come from `config/countries/KEN.toml`. Shared calculations live in `src/national_tool_metrics/sections/adaptation_potential.py`.

In [21]:
from pathlib import Path
import importlib
import sys

WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
SRC_DIRECTORY = REPO_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from national_tool_metrics import load_country_config
from national_tool_metrics.boundaries import load_admin_boundaries
from national_tool_metrics.outputs import validate_section_output, write_section_output
import national_tool_metrics.sections.adaptation_potential as adaptation_section

importlib.reload(adaptation_section)
from national_tool_metrics.sections.adaptation_potential import (
    assemble_adaptation_potential_metrics,
    build_flopros_metrics,
    build_nbs_metrics,
    build_river_network_context_metrics,
)

In [22]:
config = load_country_config("KEN", repo_root=REPO_ROOT)
admin_regions = load_admin_boundaries(config)

print(f"Country: {config.country.name} ({config.country.iso3})")
print(f"Administrative level: {config.country.admin_level.upper()}")
print(f"Administrative regions: {len(admin_regions):,}")
print(f"Nominal NbS cell area: {config.parameters['nbs_nominal_cell_area_ha']} ha")
print(f"Maximum coastal assignment distance: {config.parameters['max_coastal_assignment_distance_m']:,} m")

Country: Kenya (KEN)
Administrative level: ADM1
Administrative regions: 47
Nominal NbS cell area: 6.25 ha
Maximum coastal assignment distance: 5,000 m


## 1. Existing Flood Protection — FLOPROS

Calculate the modal positive FLOPROS return-period value in each administrative region. Zero-valued cells are treated as no-data; where more than one positive value has the same highest frequency, the lower return period is selected.

In [23]:
flopros_metrics = build_flopros_metrics(config, admin_regions)
flopros_metrics.head()

,adm_id,flopros_protection_standard_mode_rp
0,32016919B72266624462344,2.08152
1,32016919B63496705134089,2.00941
2,32016919B2031803566233,2.05484
3,32016919B89873713911655,2.05484
4,32016919B96045830258165,2.08152


## 2. Nature-based Solution Potential

Summarize slope vegetation, mangrove, and river catchment restoration opportunities. Area and per-hectare totals use the source documentation's nominal 9-arcsecond cell assumption of 6.25 hectares. Mangrove cells outside administrative polygons are assigned to their nearest region only within the configured 5 km cap. Biodiversity is averaged over valid opportunity cells; cost and carbon metrics are summed over valid opportunity cells. Slope-vegetation costs and benefits are also broken down into other land cover, crops, and bare ground; mangrove metrics are broken down into accreting, static or moderately retreating, and fast-retreating shoreline conditions. River catchment restoration remains a total because its opportunity raster is binary.

In [24]:
nbs_metrics = build_nbs_metrics(config, admin_regions)
nbs_metrics.head()

,adm_id,nbs_slope_vegetation_total_km2,nbs_slope_vegetation_other_km2,nbs_slope_vegetation_crops_km2,nbs_slope_vegetation_bare_ground_km2,nbs_mangrove_total_km2,nbs_mangrove_accreting_km2,nbs_mangrove_static_moderate_retreat_km2,nbs_mangrove_fast_retreat_km2,nbs_river_catchment_restoration_total_km2,...,nbs_river_catchment_restoration_carbon_benefit_total_tonnes,nbs_slope_vegetation_biodiversity_benefit_mean,nbs_slope_vegetation_other_biodiversity_benefit_mean,nbs_slope_vegetation_crops_biodiversity_benefit_mean,nbs_slope_vegetation_bare_ground_biodiversity_benefit_mean,nbs_mangrove_biodiversity_benefit_mean,nbs_mangrove_accreting_biodiversity_benefit_mean,nbs_mangrove_static_moderate_retreat_biodiversity_benefit_mean,nbs_mangrove_fast_retreat_biodiversity_benefit_mean,nbs_river_catchment_restoration_biodiversity_benefit_mean
0,32016919B72266624462344,5.5,5.5000,0.0000,0.0,0.0,0.0,0.0,0.0,8.1250,...,397943.75,0.551404,0.551404,0.000000,0.0,0.0,0.0,0.0,0.0,0.554082
1,32016919B63496705134089,0.0,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,...,0.00,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
2,32016919B2031803566233,0.0,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,...,0.00,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
3,32016919B89873713911655,0.0,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,...,0.00,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
4,32016919B96045830258165,567.0,127.8125,439.1875,0.0,0.0,0.0,0.0,0.0,419.5625,...,20789806.25,0.618275,0.568507,0.632759,0.0,0.0,0.0,0.0,0.0,0.516755


## 3. River Network Context

Clip the river network to each administrative region and classify its length using the grid-cell urbanisation raster. Classes 10 to 13 are grouped as rural (including water), classes 21 to 23 as town, and class 30 as city. Small raster no-data gaps at boundaries use the nearest valid class within 5 km. The three classified lengths are required to sum to total river length.

In [25]:
river_network_context_metrics = build_river_network_context_metrics(
    config,
    admin_regions,
)
river_network_context_metrics.head()

,adm_id,river_length_total_km,river_length_rural_km,river_length_town_km,river_length_city_km
0,32016919B72266624462344,2593.631957,2561.732790,27.160384,4.738783
1,32016919B63496705134089,2939.552667,2935.294510,4.258157,0.000000
2,32016919B2031803566233,1142.137915,1104.284056,27.318816,10.535043
3,32016919B89873713911655,2583.677867,2563.473609,20.204258,0.000000
4,32016919B96045830258165,270.311559,260.947687,9.363872,0.000000


## 4. Combine and Validate

Create one baseline row per administrative region and validate the standardized section schema.

In [26]:
adaptation_potential_metrics = assemble_adaptation_potential_metrics(
    config,
    admin_regions,
    flopros_metrics,
    nbs_metrics,
    river_network_context_metrics,
)
validate_section_output(adaptation_potential_metrics, "adaptation_potential")

print(f"Rows: {len(adaptation_potential_metrics):,}")
print(f"Metric columns: {len(adaptation_potential_metrics.columns) - 9:,}")
adaptation_potential_metrics.head()

Rows: 47
Metric columns: 50


,country_iso3,country_name,admin_level,adm_id,adm_name,section,hazard,scenario,model_run,flopros_protection_standard_mode_rp,...,nbs_slope_vegetation_bare_ground_biodiversity_benefit_mean,nbs_mangrove_biodiversity_benefit_mean,nbs_mangrove_accreting_biodiversity_benefit_mean,nbs_mangrove_static_moderate_retreat_biodiversity_benefit_mean,nbs_mangrove_fast_retreat_biodiversity_benefit_mean,nbs_river_catchment_restoration_biodiversity_benefit_mean,river_length_total_km,river_length_rural_km,river_length_town_km,river_length_city_km
0,KEN,Kenya,ADM1,32016919B72266624462344,Turkana,adaptation_potential,none,baseline,baseline_inputs,2.082,...,0.0,0.0,0.0,0.0,0.0,0.554,2593.632,2561.733,27.160,4.739
1,KEN,Kenya,ADM1,32016919B63496705134089,Marsabit,adaptation_potential,none,baseline,baseline_inputs,2.009,...,0.0,0.0,0.0,0.0,0.0,0.000,2939.553,2935.295,4.258,0.000
2,KEN,Kenya,ADM1,32016919B2031803566233,Mandera,adaptation_potential,none,baseline,baseline_inputs,2.055,...,0.0,0.0,0.0,0.0,0.0,0.000,1142.138,1104.284,27.319,10.535
3,KEN,Kenya,ADM1,32016919B89873713911655,Wajir,adaptation_potential,none,baseline,baseline_inputs,2.055,...,0.0,0.0,0.0,0.0,0.0,0.000,2583.678,2563.474,20.204,0.000
4,KEN,Kenya,ADM1,32016919B96045830258165,West Pokot,adaptation_potential,none,baseline,baseline_inputs,2.082,...,0.0,0.0,0.0,0.0,0.0,0.517,270.312,260.948,9.364,0.000


## 5. Export

Write the canonical Adaptation Potential CSV after reviewing the table above.

In [27]:
output_path = write_section_output(
    adaptation_potential_metrics,
    config,
    "adaptation_potential",
)
print(f"Exported Adaptation Potential metrics to: {output_path}")

Exported Adaptation Potential metrics to: C:\Users\Mark.DESKTOP-UFHIN6T\Projects\national_tool_metrics\results\KEN\adaptation_potential\KEN_adm1_adaptation_potential_metrics.csv
